# Week 4: Predictive Modeling for Agriculture Applications

## Crop Yield Prediction

This notebook demonstrates a practical machine learning framework for predicting crop yield from agricultural data. It includes data loading, basic inspection, preprocessing, model training, comparison, and evaluation using regression metrics.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load the Dataset

Place the crop yield CSV inside a `data` folder. Update the filename below if your dataset has a different name.

In [ ]:
DATA_PATH = 'data/crop_yield_dataset.csv'
df = pd.read_csv(DATA_PATH)

print('Dataset shape:', df.shape)
display(df.head())

## 3. Basic Data Inspection

In [ ]:
print(df.info())
print('\nMissing values:')
display(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

## 4. Remove Duplicate Records

Duplicate rows are removed before modeling. This step should be reviewed against the original data because repeated observations can sometimes be legitimate.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
print('Shape after removing duplicates:', df.shape)

## 5. Select the Target Variable

The target should be the continuous crop-yield column. The code below first looks for common names. If your dataset uses another name, set `TARGET` manually.

In [ ]:
possible_targets = ['Yield', 'yield', 'Crop_Yield', 'crop_yield', 'Yield_ton_per_hectare', 'Crop Yield']
TARGET = next((c for c in possible_targets if c in df.columns), None)

if TARGET is None:
    raise ValueError(f'Please set TARGET manually. Available columns: {list(df.columns)}')

print('Target column:', TARGET)

## 6. Prepare Features

Identifiers are removed where possible. Numerical and categorical columns are processed separately.

In [ ]:
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()

# Remove common identifier columns if present
id_columns = [c for c in X.columns if c.lower() in ['id', 'index', 'record_id']]
X = X.drop(columns=id_columns, errors='ignore')

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print('Numerical features:', numeric_features)
print('Categorical features:', categorical_features)

## 7. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

print('Training rows:', len(X_train))
print('Testing rows:', len(X_test))

## 8. Preprocessing Pipeline

Missing numerical values are replaced with the median. Missing categorical values are replaced with the most frequent category, followed by one-hot encoding.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

## 9. Train and Compare Regression Models

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42, max_depth=10),
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(random_state=42)
}

results = []
trained_models = {}

for name, model in models.items():
    pipe = Pipeline([
        ('preprocessing', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    predictions = pipe.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append([name, mae, rmse, r2])
    trained_models[name] = pipe

results_df = pd.DataFrame(results, columns=['Model', 'MAE', 'RMSE', 'R2'])
results_df.sort_values('RMSE')

## 10. Evaluation

- **MAE:** average absolute prediction error.
- **RMSE:** gives greater weight to larger prediction errors.
- **R²:** indicates how much variation in yield is explained by the model.

Lower MAE and RMSE are generally preferred, while a higher R² indicates better explanatory performance. Final model selection should also consider validation results and practical requirements.

In [ ]:
best_model_name = results_df.sort_values('RMSE').iloc[0]['Model']
print('Model with the lowest test RMSE:', best_model_name)
display(results_df.sort_values('RMSE'))

## 11. Example Prediction

The trained model can be used to predict yield for new observations with the same feature structure as the training data.

In [ ]:
best_model = trained_models[best_model_name]
sample_predictions = best_model.predict(X_test.head(5))

comparison = pd.DataFrame({
    'Actual Yield': y_test.head(5).values,
    'Predicted Yield': sample_predictions
})
display(comparison)

## 12. Conclusion

This notebook provides a practical baseline framework for crop yield prediction. The workflow covers data preparation, preprocessing, regression model comparison, and evaluation. For a production system, time-aware validation, additional weather and soil variables, hyperparameter tuning, and external validation across regions and years should be added.